# BODAQS Macro Analysis Dashboard

This notebook drives the extracted backend (`bodaqs_analysis`) with minimal editing.

Workflow:
1. Load CSV
2. Normalize (zero + scale)
3. Estimate velocity/acceleration
4. Load event schema + detect events
5. Inspect metrics / quick plots


In [1]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path

from bodaqs_analysis import run_macro
from bodaqs_analysis import normalize_and_scale
from bodaqs_analysis import estimate_va
from bodaqs_analysis import load_event_schema
from bodaqs_analysis import detect_events_from_schema
from bodaqs_analysis import extract_metrics_df

import importlib
import bodaqs_analysis.widgets.event_browser as eb
importlib.reload(eb)

from bodaqs_analysis.widgets.event_browser import make_event_browser_widget_for_loader


logging.basicConfig(
    level=logging.DEBUG,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)


In [2]:
# ---- Inputs ----
CSV_PATH = r"C:\Users\Ben Connor\OneDrive\Documents\logs\2026-01-04_06-37-32.CSV"   # change to your log
SCHEMA_PATH = r"event schema\event_schema.yaml"

# Columns and full-ranges (mm) for normalization (edit as needed)
NORMALIZE_RANGES = {
    "front_shock_dom_suspension [mm]": 170.0,
    "rear_shock_dom_suspension [mm]": 55.0,
}

SAMPLE_RATE_HZ = 200   # if known; used for VA estimation when time col is absent


In [3]:
# ---- Batch orchestration ----

pattern = Path(CSV_PATH)
CSV_FILES = sorted(pattern.parent.glob(pattern.name))

if not CSV_FILES:
    raise FileNotFoundError(f"No CSV files matched: {CSV_PATH}")

logger.info(f"Found {len(CSV_FILES)} files:")
for p in CSV_FILES:
    logger.info("  %s", p.name)

events_all = []
metrics_all = []
results_all = []  # optional if you want to inspect per-file outputs
sessions_by_id = {}
schema = None

for p in CSV_FILES:
    logger.info(f"Processing {p.name} ...")
    results = run_macro(
        str(p),
        SCHEMA_PATH,
        normalize_ranges=NORMALIZE_RANGES,
        sample_rate_hz=SAMPLE_RATE_HZ,
    )
    session = results["session"]
    sid = str(session["session_id"])
    sessions_by_id[sid] = session
    #results_all.append((p, results))  # optional

    # pull from the correct keys
    ev = results["events"]
    mt = results["metrics"]
    if schema is None:
        schema = results["schema"]
    events_df = results["events"]

    # only append if non-empty (optional, but handy)
    if isinstance(ev, pd.DataFrame) and not ev.empty:
        events_all.append(ev)
    if isinstance(mt, pd.DataFrame) and not mt.empty:
        metrics_all.append(mt)

events_df  = pd.concat(events_all, ignore_index=True) if events_all else pd.DataFrame()
metrics_df = pd.concat(metrics_all, ignore_index=True) if metrics_all else pd.DataFrame()

logging.debug("events_df: %s", events_df.shape)
logging.debug("metrics_df: %s", metrics_df.shape)






INFO:__main__:Found 1 files:
INFO:__main__:  2026-01-04_06-37-32.CSV
INFO:__main__:Processing 2026-01-04_06-37-32.CSV ...
INFO:bodaqs_analysis.pipeline:Session load complete: C:\Users\Ben Connor\OneDrive\Documents\logs\2026-01-04_06-37-32.CSV
INFO:bodaqs_analysis.pipeline:Session pre-process complete
DEBUG:bodaqs_analysis.pipeline:time_s start/end: 0.002 .. 283.292
DEBUG:bodaqs_analysis.pipeline:dt median/min/max: 0.0020000000000024443 / 0.0019999999999527063 / 0.0020000000000095497
DEBUG:bodaqs_analysis.pipeline:signals entries: 12
DEBUG:bodaqs_analysis.pipeline:front_shock_dom_suspension [mm] -> {'kind': '', 'unit': 'mm', 'domain': 'suspension', 'op_chain': [], 'sensor': 'front_shock', 'quantity': 'disp'}
DEBUG:bodaqs_analysis.pipeline:front_shock_raw_dom_suspension [counts] -> {'kind': 'raw', 'unit': 'counts', 'domain': 'suspension', 'op_chain': [], 'sensor': 'front_shock', 'quantity': 'raw'}
DEBUG:bodaqs_analysis.pipeline:rear_shock_dom_suspension [mm] -> {'kind': '', 'unit': 'mm',

front_shock_dom_suspension [mm]
front_shock_raw_dom_suspension [counts]
front_shock_dom_suspension [mm]_op_zeroed
front_shock_dom_suspension [1]_op_zeroed_norm
front_shock_vel_dom_suspension [mm/s]
front_shock_acc_dom_suspension [mm/s^2]


DEBUG:bodaqs_analysis.detect:deep_compression(rear_shock): candidate t0_idx=105492 failed conditions
DEBUG:bodaqs_analysis.detect:deep_compression(rear_shock): candidate t0_idx=105691 failed conditions
DEBUG:bodaqs_analysis.detect:deep_compression(rear_shock): candidate t0_idx=106768 failed conditions
DEBUG:bodaqs_analysis.detect:deep_compression(rear_shock): candidate t0_idx=107213 failed conditions
DEBUG:bodaqs_analysis.detect:deep_compression(rear_shock): candidate t0_idx=107471 failed conditions
DEBUG:bodaqs_analysis.detect:deep_compression(rear_shock): candidate t0_idx=109269 failed conditions
DEBUG:bodaqs_analysis.detect:deep_compression(rear_shock): candidate t0_idx=109713 failed conditions
DEBUG:bodaqs_analysis.detect:deep_compression(rear_shock): candidate t0_idx=110106 failed conditions
DEBUG:bodaqs_analysis.detect:deep_compression(rear_shock): candidate t0_idx=110332 failed conditions
DEBUG:bodaqs_analysis.detect:deep_compression(rear_shock): candidate t0_idx=110494 failed c

signals count: 12
example disp cols: ['front_shock_dom_suspension [mm]', 'rear_shock_dom_suspension [mm]', 'front_shock_dom_suspension [mm]_op_zeroed', 'front_shock_dom_suspension [1]_op_zeroed_norm', 'rear_shock_dom_suspension [mm]_op_zeroed', 'rear_shock_dom_suspension [1]_op_zeroed_norm']
example keys: ['kind', 'unit', 'domain', 'op_chain', 'sensor', 'quantity']
disp_norm prefer: {'quantity': 'disp', 'unit': '1', 'op_chain': ['zeroed', 'norm']}
DEBUG disp_norm prefer: {'quantity': 'disp', 'unit': '1', 'op_chain': ['zeroed', 'norm']}
DEBUG disp_norm pref_sensor: None
DEBUG disp_norm pref_quantity: disp
DEBUG disp_norm pref_unit: 1
DEBUG disp_norm pref_op_chain: ('zeroed', 'norm')
DEBUG disp_norm candidate: front_shock_dom_suspension [mm] sensor= front_shock unit= mm op_chain= []
DEBUG disp_norm candidate: rear_shock_dom_suspension [mm] sensor= rear_shock unit= mm op_chain= []
DEBUG disp_norm candidate: front_shock_dom_suspension [mm]_op_zeroed sensor= front_shock unit= mm op_chain= [

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output


def metric_histogram_widget_v5(
    events_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    *,
    event_type_col: str = "event_name",
    signal_col: str = "signal_col",
    default_bins: int = 10,
    max_bins: int = 200,
):
    logging.debug("events_df shape: %s", events_df.shape)
    logging.debug("metrics_df shape: %s", metrics_df.shape)

    for col in ("session_id", "event_id", event_type_col, signal_col):
        if col not in events_df.columns:
            raise ValueError(f"events_df missing required column: {col}")
    for col in ("session_id", "event_id"):
        if col not in metrics_df.columns:
            raise ValueError(f"metrics_df missing required column: {col}")

    metric_cols = [c for c in metrics_df.columns if isinstance(c, str) and c.startswith("m_")]
    if not metric_cols:
        raise ValueError("No metric columns found in metrics_df (expected 'm_*')")

    viz_df = events_df[["session_id", "event_id", event_type_col, signal_col]].merge(
        metrics_df[["session_id", "event_id"] + metric_cols],
        on=["session_id", "event_id"],
        how="inner",
        validate="one_to_one",
    )
    print("joined viz_df shape:", viz_df.shape)

    sessions = sorted([x for x in viz_df["session_id"].dropna().unique()])
    event_types = sorted([x for x in viz_df[event_type_col].dropna().unique()])
    signals = sorted([x for x in viz_df[signal_col].dropna().unique()])
    metrics = sorted(metric_cols)

    if not sessions:
        raise ValueError("No non-null session_id values after join")
    if not event_types:
        raise ValueError(f"No non-null values found in {event_type_col} after join")
    if not signals:
        raise ValueError(f"No non-null values found in {signal_col} after join")

    # --- widgets ---
    w_sess_mode = W.RadioButtons(
        options=[("Aggregate sessions", False), ("Compare sessions", True)],
        value=False,
        description="Sessions:",
    )
    w_sessions = W.SelectMultiple(
        options=sessions,
        value=tuple(sessions[:1]),
        description="Pick:",
        rows=min(8, max(3, len(sessions))),
    )

    w_sig_mode = W.RadioButtons(
        options=[("Aggregate signals", False), ("Compare signals", True)],
        value=False,
        description="Signals:",
    )
    w_signals = W.SelectMultiple(
        options=signals,
        value=tuple(signals[:1]),
        description="Pick:",
        rows=min(8, max(3, len(signals))),
    )

    w_event = W.Dropdown(options=event_types, value=event_types[0], description="Event:")
    w_metric = W.Dropdown(options=metrics, value=metrics[0], description="Metric:")
    w_bins = W.BoundedIntText(value=int(default_bins), min=1, max=int(max_bins), step=1, description="Bins:")
    w_cdf = W.Checkbox(value=False, description="CDF (cumulative)")
    w_norm = W.Checkbox(value=True, description="Normalize (proportion)")

    out = W.Output()

    def _series_stats(name: str, vals: np.ndarray) -> str:
        if len(vals) == 0:
            return f"- {name}: count=0"
        vmin = float(np.min(vals))
        vmax = float(np.max(vals))
        mean = float(np.mean(vals))
        med = float(np.median(vals))
        return f"- {name}: count={len(vals)}  min={vmin:.6g}  max={vmax:.6g}  mean={mean:.6g}  median={med:.6g}"

    def _plot_series(ax, vals: np.ndarray, bins: int, *, label: str | None):
        if len(vals) == 0:
            return
        if w_cdf.value:
            x = np.sort(vals)
            y = np.arange(1, len(x) + 1, dtype=float)
            if w_norm.value:
                y = y / float(len(x))
            ax.step(x, y, where="post", label=label)
        else:
            weights = None
            if w_norm.value:
                weights = np.ones_like(vals, dtype=float) / float(len(vals))
            ax.hist(vals, bins=int(bins), weights=weights, histtype="step", label=label)

    def _render(*_):
        with out:
            clear_output(wait=True)

            sel_sessions = list(w_sessions.value)
            sel_signals = list(w_signals.value)

            if not sel_sessions:
                print("Select at least one session.")
                return
            if not sel_signals:
                print("Select at least one signal.")
                return

            base = viz_df[
                (viz_df[event_type_col] == w_event.value) &
                (viz_df["session_id"].isin(sel_sessions)) &
                (viz_df[signal_col].isin(sel_signals))
            ]

            # Decide series breakdown
            compare_sessions = bool(w_sess_mode.value)
            compare_signals = bool(w_sig_mode.value)

            series = []

            if compare_sessions and compare_signals:
                # One series per (session, signal)
                for sid in sel_sessions:
                    for sig in sel_signals:
                        sub = base[(base["session_id"] == sid) & (base[signal_col] == sig)]
                        vals = pd.to_numeric(sub[w_metric.value], errors="coerce").dropna().to_numpy()
                        series.append((f"{sid} | {sig}", vals))

            elif compare_sessions and (not compare_signals):
                # One series per session (signals aggregated within session)
                for sid in sel_sessions:
                    sub = base[base["session_id"] == sid]
                    vals = pd.to_numeric(sub[w_metric.value], errors="coerce").dropna().to_numpy()
                    series.append((str(sid), vals))

            elif (not compare_sessions) and compare_signals:
                # One series per signal (sessions aggregated within signal)
                for sig in sel_signals:
                    sub = base[base[signal_col] == sig]
                    vals = pd.to_numeric(sub[w_metric.value], errors="coerce").dropna().to_numpy()
                    series.append((str(sig), vals))

            else:
                # Fully aggregated: sessions + signals all pooled
                vals = pd.to_numeric(base[w_metric.value], errors="coerce").dropna().to_numpy()
                series.append(("aggregate", vals))

            fig, ax = plt.subplots(figsize=(8.3, 4.2))

            for name, vals in series:
                _plot_series(ax, vals, int(w_bins.value), label=(name if (compare_sessions or compare_signals) else None))

            mode_bits = []
            mode_bits.append("sessions=compare" if compare_sessions else "sessions=aggregate")
            mode_bits.append("signals=compare" if compare_signals else "signals=aggregate")

            ax.set_title(
                f"{w_metric.value} distribution\n"
                f"{event_type_col}={w_event.value} | {', '.join(mode_bits)}"
            )
            ax.set_xlabel(w_metric.value)
            if w_cdf.value:
                ax.set_ylabel("Cumulative proportion" if w_norm.value else "Cumulative count")
            else:
                ax.set_ylabel("Proportion" if w_norm.value else "Count")

            ax.grid(True, which="major", axis="both", alpha=0.3)

            if (compare_sessions or compare_signals):
                ax.legend(title="series", fontsize=9)

            if all(len(vals) == 0 for _, vals in series):
                ax.text(0.5, 0.5, "No numeric values after filtering",
                        ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()

            plt.show()

            print("Summary stats:")
            for name, vals in series:
                print(_series_stats(name, vals))

    for w in (w_sess_mode, w_sessions, w_sig_mode, w_signals, w_event, w_metric, w_bins, w_cdf, w_norm):
        w.observe(_render, names="value")

    controls = W.VBox([
        W.HBox([w_event, w_metric, w_bins, w_cdf, w_norm]),
        W.HBox([
            W.VBox([w_sess_mode, w_sessions]),
            W.VBox([w_sig_mode, w_signals]),
        ])
    ])

    display(W.VBox([controls, out]))
    _render()

    #return {"viz_df": viz_df, "out": out}
    return {}

# Call explicitly
metric_histogram_widget_v5(
    events_df,
    metrics_df,
    event_type_col="event_name",
    signal_col="signal_col",
    default_bins=10,
)


DEBUG:root:events_df shape: (63, 35)
DEBUG:root:metrics_df shape: (63, 19)


joined viz_df shape: (63, 8)


{}

In [5]:
from bodaqs_analysis.widgets.event_browser import make_event_browser_widget_for_loader

def session_loader(session_id: str) -> dict:
    return sessions_by_id[str(session_id)]

wb = make_event_browser_widget_for_loader(
    schema,
    events_df,
    session_loader=session_loader,
)
